In [4]:
!pip install -q "google-genai>=2.12.1,<3"

In [10]:

import pandas as pd
import json
from getpass import getpass
from google import genai
from google.genai import types

# Enter your replacement API key privately
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

# Load follow-up notes
df = pd.read_csv("followup_notes.csv")
print("Notes loaded:", len(df))

# Prepare notes for Gemini
notes = df[["note_id", "appointment_id", "note_text"]].fillna("").to_dict("records")

prompt = f"""
Analyze these patient appointment follow-up notes.

For each note, identify:
1. reason_category: Choose one of
   Forgetfulness, Transport, Financial, Illness, Work,
   Unreachable, Family Emergency, Other, No Information.
2. sentiment: Positive, Neutral, or Negative.
3. urgency: High, Medium, or Low.
4. recommended_action: A short practical action the clinic should take.

Return ONLY a valid JSON array.
Each object must contain:
note_id, appointment_id, reason_category,
sentiment, urgency, recommended_action.

Do not invent information. Use No Information
when the note is blank.

Notes:
{json.dumps(notes, ensure_ascii=False)}
"""

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.2
    )
)

# Convert Gemini response into a DataFrame
results = json.loads(response.text)
analysed_df = pd.DataFrame(results)

# Save final output
analysed_df.to_csv("notes_analysed.csv", index=False)

print("Analysis completed!")
print("Records analysed:", len(analysed_df))
display(analysed_df.head(10))

print("Exported notes_analysed.csv")


Enter your Gemini API key: ··········
Notes loaded: 30
Analysis completed!
Records analysed: 30


,note_id,appointment_id,reason_category,sentiment,urgency,recommended_action
0,N001,A0872,Forgetfulness,Neutral,Medium,Reschedule for next week and verify reminder s...
1,N002,A0897,Transport,Negative,Low,Suggest tele-consultation or a closer clinic f...
2,N003,A0526,Financial,Negative,Medium,Discuss payment plans or follow-up discounts w...
3,N004,A0926,No Information,Neutral,Low,Attempt to contact patient to determine reason...
4,N005,A0534,Illness,Neutral,Medium,Follow up in 3 days to check recovery and resc...
5,N006,A0507,Work,Neutral,Medium,Reschedule for an evening slot as requested.
6,N007,A0333,Forgetfulness,Neutral,Medium,Reschedule appointment and verify SMS notifica...
7,N008,A0222,Transport,Negative,Low,Offer tele-health options to mitigate distance...
8,N009,A0977,Unreachable,Neutral,Low,Try reaching out via alternative contact metho...
9,N010,A0596,Work,Neutral,Medium,Reschedule for a Saturday slot as per patient ...


Exported notes_analysed.csv
